In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# Task 1: Data Preparation
# ==========================================
print("Task 1: Loading and Preprocessing SVHN Data...")

# SVHN images are 3x32x32 RGB images. We normalize across all 3 channels.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) 
])

# Note: SVHN uses 'split' instead of 'train=True/False'
train_data = datasets.SVHN(root='./data', split='train', download=True, transform=transform)
test_data = datasets.SVHN(root='./data', split='test', download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

# ==========================================
# Task 2: Architecture Design
# ==========================================
class SVHN_CNN(nn.Module):
    def __init__(self):
        super(SVHN_CNN, self).__init__()
        
        # Convolutional Block 1
        # Input: 3 channels (RGB) | Output: 16 channels | Image size: 32x32
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # Reduces to 16x16
        
        # Convolutional Block 2
        # Input: 16 channels | Output: 32 channels | Image size: 16x16
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) # Reduces to 8x8
        
        # Fully Connected Block
        self.flatten = nn.Flatten()
        # 32 channels * 8 width * 8 height
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.dropout = nn.Dropout(0.5) # 50% probability to drop a neuron
        self.fc2 = nn.Linear(128, 10)  # 10 output classes (digits 0-9)

    def forward(self, x):
        # Block 1: Conv -> BatchNorm -> ReLU -> Pool
        x = self.pool1(torch.relu(self.bn1(self.conv1(x))))
        # Block 2: Conv -> BatchNorm -> ReLU -> Pool
        x = self.pool2(torch.relu(self.bn2(self.conv2(x))))
        
        # Flatten and FC layers
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = SVHN_CNN()

# ==========================================
# Task 3: Training & Regularization
# ==========================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5
print(f"\nTask 3: Starting training for {epochs} epochs...")

for epoch in range(epochs):
    model.train() # Set to training mode (enables Dropout and BatchNorm tracking)
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        # Calculate accuracy for this batch
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

# ==========================================
# Task 4: Error Analysis (Visual Debugging)
# ==========================================
print("\nTask 4: Finding misclassified images for error analysis...")
model.eval() # Set to evaluation mode (disables Dropout)

# Lists to store the incorrect examples
wrong_images = []
true_labels = []
pred_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        # Create a boolean mask of where predictions don't match labels
        mismatches = (predicted != labels)
        
        # Extract the specific mismatched items from this batch
        for i in range(len(mismatches)):
            if mismatches[i] and len(wrong_images) < 5:
                wrong_images.append(images[i])
                true_labels.append(labels[i].item())
                pred_labels.append(predicted[i].item())
                
        if len(wrong_images) >= 5:
            break

# Un-normalize function for display purposes
def imshow(img):
    img = img / 2 + 0.5 # unnormalize: (x * std) + mean
    npimg = img.numpy()
    # PyTorch is (Channels, Height, Width) but Matplotlib is (Height, Width, Channels)
    plt.imshow(np.transpose(npimg, (1, 2, 0)))

# Plotting the 5 errors
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.suptitle('Visual Debugging: Misclassified SVHN Images', fontsize=16)

for i in range(5):
    plt.sca(axes[i])
    imshow(wrong_images[i])
    axes[i].set_title(f"True: {true_labels[i]}\nPred: {pred_labels[i]}", color='red')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras import layers, models

# ==========================================
# Task 1: Data Preparation
# ==========================================
print("Task 1: Loading and Preprocessing SVHN Data...")

# Load SVHN dataset from TensorFlow Datasets
# as_supervised=True returns a tuple (image, label) instead of a dictionary
(ds_train, ds_test), ds_info = tfds.load(
    'svhn_cropped', 
    split=['train', 'test'], 
    shuffle_files=True, 
    as_supervised=True, 
    with_info=True
)

# Preprocessing function to match the PyTorch Normalization
def preprocess(image, label):
    # Convert from uint8 [0, 255] to float32
    image = tf.cast(image, tf.float32) / 255.0
    # Normalize to [-1, 1] using (x - mean) / std where mean=0.5, std=0.5
    image = (image - 0.5) / 0.5 
    return image, label

# Map the preprocessing, set batch size, and prefetch for performance
BATCH_SIZE = 64
ds_train = ds_train.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
ds_train = ds_train.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

ds_test = ds_test.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# ==========================================
# Task 2: Architecture Design
# ==========================================
print("\nTask 2: Initializing Keras Model Architecture...")

# Building the exact equivalent of the PyTorch SVHN_CNN
model = models.Sequential([
    # Input definition: SVHN images are 32x32 pixels with 3 RGB channels
    layers.InputLayer(shape=(32, 32, 3)),
    
    # Convolutional Block 1
    layers.Conv2D(16, kernel_size=(3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(pool_size=(2, 2)), # Reduces to 16x16
    
    # Convolutional Block 2
    layers.Conv2D(32, kernel_size=(3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(pool_size=(2, 2)), # Reduces to 8x8
    
    # Fully Connected Block
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3), # 50% probability to drop a neuron
    
    # Output Layer: 10 classes (digits 0-9)
    # We leave activation blank (Linear) and handle Softmax in the loss function
    layers.Dense(10)
])

model.summary()

# ==========================================
# Task 3: Training & Regularization
# ==========================================
print("\nTask 3: Starting training for 5 epochs...")

# Compile the model
# from_logits=True because our final Dense layer doesn't have a Softmax activation
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# Keras handles the training loop internally with model.fit()
# Validation data is passed so we can see test accuracy per epoch
history = model.fit(
    ds_train, 
    epochs=5, 
    validation_data=ds_test
)

# ==========================================
# Task 4: Error Analysis (Visual Debugging)
# ==========================================
print("\nTask 4: Finding misclassified images for error analysis...")

wrong_images = []
true_labels_list = []
pred_labels_list = []

# Unbatch the test dataset and grab a large chunk to guarantee we find 5 errors
for images, labels in ds_test.unbatch().batch(500).take(1):
    # Get raw predictions
    predictions = model.predict(images, verbose=0)
    # Convert raw logits to specific class predictions (0-9)
    pred_classes = np.argmax(predictions, axis=1)
    truth = labels.numpy()
    
    # Create a boolean mask of mismatches
    mismatches = (pred_classes != truth)
    
    for i in range(len(mismatches)):
        if mismatches[i] and len(wrong_images) < 5:
            wrong_images.append(images[i].numpy())
            true_labels_list.append(truth[i])
            pred_labels_list.append(pred_classes[i])
            
        if len(wrong_images) >= 5:
            break

# Un-normalize function for Matplotlib display
def unnormalize(img):
    img = img * 0.5 + 0.5 # reverse the (img - 0.5)/0.5 operation
    return np.clip(img, 0, 1)

# Plotting the 5 errors
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.suptitle('Visual Debugging: Misclassified SVHN Images (TensorFlow/Keras)', fontsize=16)

for i in range(5):
    plt.sca(axes[i])
    # Matplotlib natively handles (Height, Width, Channels) which matches TF's default
    plt.imshow(unnormalize(wrong_images[i]))
    axes[i].set_title(f"True: {true_labels_list[i]}\nPred: {pred_labels_list[i]}", color='red')
    axes[i].axis('off')

plt.tight_layout()
plt.show()